# 01 · Limpieza: Base de datos Hackaton_anonimizada

Base de empresas afiliadas a ACTHO exportada del CRM. Una fila = un registro de empresa/afiliado.

- Entrada: `datos/originales/Base de datos Hackaton_anonimizada.xlsx`
- Salida: `datos/limpios/01_base_hackaton_limpio.csv` y `datos/limpios/01_diccionario_base_hackaton.csv`

Criterio general: **no se borran filas**. Los problemas se marcan con columnas de bandera para que cada análisis decida qué hacer.

In [ ]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

## 1. Carga
Una sola hoja, encabezados en la primera fila. Se guarda una copia del original para comparar al final.

In [ ]:
df = pd.read_excel(ORIGINALES / "Base de datos Hackaton_anonimizada.xlsx")
df_original = df.copy()
assert df.shape == (1073, 49), df.shape
print(f"Original: {df.shape[0]} filas x {df.shape[1]} columnas")

## 2. Nombres de columnas
Minúsculas, guiones bajos y sin símbolos (`%` → `pct`), para poder escribirlos fácil en el código.

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
              .str.replace("%", "pct", regex=False)
              .str.replace(" ", "_")
              .str.replace(r"[^\w]", "", regex=True))
df = df.rename(columns={"pct_de_mujeres_en_la_compañía": "pct_mujeres_en_la_compañía"})

## 3. Texto dañado y espacios
580 celdas traen tildes dañadas por un problema de codificación al exportar: `Bogotﾃ｡` en vez de `Bogotá`, `Estﾃ｡ndar` en vez de `Estándar`.
Son letras UTF-8 leídas como japonés (cp932); se reparan deshaciendo esa lectura. Solo se toca el texto que tiene esos caracteres raros.
Luego se quitan espacios sobrantes y los textos vacíos pasan a nulo.

In [ ]:
DANADO = re.compile("[\uff61-\uff9f]")  # katakana de ancho medio: señal de tilde dañada


def reparar(s):
    return s.encode("cp932").decode("utf-8") if isinstance(s, str) and DANADO.search(s) else s


for c in df.select_dtypes(include=["object", "string"]).columns:
    df[c] = (df[c].map(reparar).astype("string").str.strip()
                  .str.replace(r"\s+", " ", regex=True)
                  .replace("", pd.NA))

assert not df.select_dtypes("string").apply(lambda s: s.str.contains(DANADO).any()).any()
df["ciudad"].value_counts().head()

## 4. Columnas sin información
Se eliminan las columnas 100% vacías (la mayoría se vaciaron al anonimizar) y `unnamed_12`, que solo tiene `False`.

In [ ]:
cols_vacias = df.columns[df.isna().all()].tolist()
print(f"Columnas 100% vacías eliminadas ({len(cols_vacias)}):", cols_vacias)
df = df.drop(columns=cols_vacias)

assert set(df["unnamed_12"].unique()) == {False}
df = df.drop(columns="unnamed_12")

## 5. Fechas
Vienen en formato ISO (`2020-03-16`), así que se convierten directo. Si alguna no se puede convertir, se avisa cuántas.

In [ ]:
cols_fecha = ["hora_de_la_última_actividad", "cambiar_hora_de_registro",
              "fecha_de_afiliación", "fecha_de_expiración_afiliación", "miembro_desde_el"]
for c in cols_fecha:
    antes = df[c].isna().sum()
    df[c] = pd.to_datetime(df[c], format="ISO8601", errors="coerce")
    perdidos = df[c].isna().sum() - antes
    if perdidos > 0:
        print(f"⚠️ {c}: {perdidos} valores no se pudieron convertir a fecha")

## 6. Porcentaje de mujeres y cantidad de empleados
- `pct_mujeres`: viene como texto (`50%`). Se pasa a número en escala 0–100.
- `cantidad_de_empleados`: el mismo rango aparece escrito de dos formas (`1-50` y `1 - 50`). Se quitan los espacios alrededor del guion.

In [ ]:
col = "pct_mujeres_en_la_compañía"
df[col] = pd.to_numeric(df[col].str.replace("%", "", regex=False).str.replace(",", ".", regex=False),
                        errors="coerce")
assert df[col].dropna().between(0, 100).all()

df["cantidad_de_empleados"] = df["cantidad_de_empleados"].str.replace(r"\s*-\s*", "-", regex=True)
df["cantidad_de_empleados"].value_counts()

## 7. Programas (vacío = no participa)
Las columnas de programas **no son 0/1**: traen cuántas veces o cuántas personas participan (valores de 1 a 10).
Vacío significa que no participa, así que se rellena con 0 y se dejan como enteros. Así se pueden sumar y filtrar (`> 0` = participa).

In [ ]:
cols_prog = ["hr_estratégico", "mas_que_un_salario", "talento_que_elige",
             "gente_que_crece", "mujeres_hr", "influenciador"]
df[cols_prog] = df[cols_prog].fillna(0).astype("Int64")
df[cols_prog].agg(["max", "sum"])

## 8. Llaves de unión
- `id_de_registro_anonimizado`: llave única del registro, en mayúsculas y sin espacios.
- `empresa_clave`: nombre de la empresa sin tildes, sin signos y sin `SAS`, `LTDA`, etc. Sirve para detectar la misma empresa escrita distinto.

In [ ]:
def sin_tildes(s):
    if pd.isna(s):
        return s
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()


llave = "id_de_registro_anonimizado"
df[llave] = df[llave].str.strip().str.upper()


def clave_nombre(s):
    if pd.isna(s):
        return s
    s = sin_tildes(s).upper()
    s = re.sub(r"[^\w\s]", " ", s)
    s = re.sub(r"\b(S A S|SAS|S A|SA|LTDA|SOCIEDAD|EMPRESA|CIA|COMPANIA)\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()


df["empresa_clave"] = df["nombre_de_empresa_anonimizado"].apply(clave_nombre)

## 9. Categorías normalizadas
Para cada columna de categoría se crea una copia `_norm` sin tildes y en formato Título (`Bogota`, `Bogotá` → `Bogota`), útil para agrupar.
La columna original se conserva tal cual.

In [ ]:
cols_cat = ["ciudad", "departamento", "ciudad_de_facturación", "país_de_facturación", "región",
            "estado", "estado_de_la_membresía", "tipo_de_membresía", "categoría_de_membresía",
            "sector_económico", "subsector_económico_anonimizado", "tipo_de_empresa",
            "tipo_de_compañía", "tipo_de_consultora"]
for c in cols_cat:
    df[c + "_norm"] = df[c].apply(lambda x: sin_tildes(x).title() if pd.notna(x) else x)
df["ciudad_norm"].value_counts().head()

## 10. Banderas de calidad (no borran filas)
- `empresa_repetida`: la misma `empresa_clave` aparece en más de un registro.
- `fecha_expiracion_invalida`: la afiliación vence antes de empezar.
- `fecha_afiliacion_futura`: afiliación posterior a la fecha de corte de la base (hay una en 2066, error de digitación).

In [ ]:
CORTE = df["hora_de_la_última_actividad"].max().normalize() + pd.Timedelta(days=1)

df["empresa_repetida"] = df.duplicated("empresa_clave", keep=False) & df["empresa_clave"].notna()
df["fecha_expiracion_invalida"] = df["fecha_de_expiración_afiliación"] < df["fecha_de_afiliación"]
df["fecha_afiliacion_futura"] = df["fecha_de_afiliación"] > CORTE
df[["empresa_repetida", "fecha_expiracion_invalida", "fecha_afiliacion_futura"]].sum()

## 11. Validaciones

In [ ]:
assert df[llave].notna().all(), "La llave tiene nulos"
assert df[llave].is_unique, "La llave tiene duplicados"
assert df.duplicated().sum() == 0, "Hay filas completamente duplicadas"
assert len(df) == len(df_original), "Se perdieron filas"

print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]} (antes: {df_original.shape[1]})")
print("Rango afiliación:", df["fecha_de_afiliación"].min(), "->", df["fecha_de_afiliación"].max())

## 12. Guardar
CSV en `utf-8-sig` para que Excel y Looker Studio lean bien las tildes. Se guarda además un diccionario con el tipo, nulos y valores únicos de cada columna.

In [ ]:
salida = LIMPIOS / "01_base_hackaton_limpio.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")

diccionario = pd.DataFrame({
    "columna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "pct_nulos": (df.isna().mean() * 100).round(1).values,
    "unicos": df.nunique().values,
})
diccionario.to_csv(LIMPIOS / "01_diccionario_base_hackaton.csv", index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == df.shape
salida.name